In [3]:
import numpy as np
from osgeo import gdal, gdal_array

fn = r"C:\Users\tanzhenyu\Dataware\GeoPy\06\XiAn-202108-AOI.tif"
# 第一种方法：使用GDAL读取数据，然后将Dataset转为NDArray
ds: gdal.Dataset = gdal.Open(fn)
print(type(ds))
im: np.ndarray = ds.ReadAsArray()
print(type(im), im.shape)

# 只读取某一个波段的数据
im = ds.GetRasterBand(1).ReadAsArray() # 读取第一个波段的数据
print(type(im), im.shape)
del ds

# 第二种方法：直接读取影像为NDArray
im = gdal_array.LoadFile(fn)
print(type(im), im.shape)

<class 'osgeo.gdal.Dataset'>
<class 'numpy.ndarray'> (4, 3938, 6677)
<class 'numpy.ndarray'> (3938, 6677)
<class 'numpy.ndarray'> (4, 3938, 6677)


In [6]:
import numpy as np
from osgeo import gdal

# 将NDArray存储为影像文件
def save2img(fname, array, driver='GTiff', prototype=None,
             transform=None, projection=None, nodata=None):
    """
    将ndarray数组写入到文件中
    :param fname: 文件路径
    :param array: NDArray类型的数组
    :param driver: 文件格式驱动
    :param prototype: 文件原型
    :param transform: GDAL中的空间转换六参数
    :param projection: 数据的投影信息
    :param nodata: NoData元数据
    """
    # 创建要写入的数据集（这里假设只有一个波段）
    # 分两种情况：一种给定了数据原型，一种没有给定，需要手动指定Transform和Projection
    driver = gdal.GetDriverByName(driver)
    if prototype:
        ds = driver.CreateCopy(fname, prototype)
    else:
        dtype = gdal.GDT_Float32
        xsize = array.shape[-1]  # 数组的列数
        ysize = array.shape[-2]  # 数组的行数
        ds = driver.Create(fname, xsize, ysize, 1, dtype)  # 这里的1指的是一个波段
        ds.SetGeoTransform(transform)
        ds.SetProjection(projection)
    # 将array写入文件
    ds.GetRasterBand(1).WriteArray(array)
    if nodata is not None:
        ds.GetRasterBand(1).SetNoDataValue(nodata)
    ds.FlushCache()
    del ds
    return fname

# 案例测试
fn = r"C:\Users\tanzhenyu\Dataware\GeoPy\06\XiAn-202108-AOI.tif"
# 将原始数据中的第三个波段进行保存
ds = gdal.Open(fn)
band = ds.GetRasterBand(3)
transform = ds.GetGeoTransform()
projection = ds.GetProjection()
nodata = band.GetNoDataValue()
im = band.ReadAsArray()
dst = r"C:\Users\tanzhenyu\Dataware\GeoPy\06\XiAn-202108-B3.tif"
save2img(dst, im, transform=transform, projection=projection, nodata=nodata)

'C:\\Users\\tanzhenyu\\Dataware\\GeoPy\\06\\XiAn-202108-B3.tif'

In [33]:
import numpy as np
from numpy import ma
from osgeo import gdal

gdal.UseExceptions()

# 使用我们进行波段叠加的多波段数据进行栅格计算
fn = '/Users/tanzhenyu/Dataware/GeoPy/06/XiAn-202108-AOI.tif'
# 进行NDVI计算(NIR – Red) / (NIR + Red)
ds = gdal.Open(fn)
# 读取数据的NoData元数据用于对计算结果掩模(这里NoData为0)
nodata = ds.GetRasterBand(1).GetNoDataValue()
red = ds.GetRasterBand(3).ReadAsArray().astype(float)
nir = ds.GetRasterBand(4).ReadAsArray().astype(float)
mask = np.logical_or(red == nodata, nir == nodata)

# 除数加上一个很小的数防止除零错误
ndvi = (nir - red) / (nir + red + 1e-8)
# 将结果规范到(-1, 1)区间内
ndvi = np.clip(ndvi, -1, 1)
# 将NoData的区域仍然设置为-9999
ndvi[mask] = -9999

# 查看结果是否符合预期
print(ma.min(ma.MaskedArray(ndvi, mask=(ndvi == -9999))), 
      ma.max(ma.MaskedArray(ndvi, mask=(ndvi == -9999))))
print(ndvi[3500: 3520, 1000: 1020])
del ds

-0.1558935361216007 0.6562736110409872
[[0.57900266 0.5802272  0.57748655 0.58221107 0.55570808 0.55506564
  0.5685187  0.57158977 0.55548998 0.56201958 0.56948897 0.57096019
  0.56512433 0.57279073 0.5805811  0.589239   0.52597616 0.50159701
  0.41811601 0.32482298]
 [0.59942456 0.58253295 0.51227778 0.56757097 0.58678983 0.57563998
  0.57928028 0.57153553 0.56236394 0.5682171  0.57484277 0.5838108
  0.57551551 0.5847391  0.57994271 0.56944105 0.49001034 0.44230902
  0.48216364 0.4969261 ]
 [0.60089193 0.60398876 0.52976136 0.54683716 0.59093905 0.59433542
  0.60044163 0.59665149 0.60285079 0.60293821 0.60222203 0.57860606
  0.56331543 0.57860825 0.58871718 0.57351823 0.51710461 0.51598232
  0.47888689 0.50161263]
 [0.59534198 0.60122805 0.58056752 0.56269592 0.58475457 0.59498187
  0.59764837 0.60307887 0.60361601 0.60243384 0.5916719  0.55605589
  0.53798641 0.5583524  0.56523683 0.55976461 0.53797882 0.48479609
  0.5134355  0.42793313]
 [0.58532211 0.58384697 0.59334478 0.5792969  

In [39]:
from osgeo import gdal

gdal.UseExceptions()

inputs = [
    '/Users/tanzhenyu/Dataware/GeoPy/06/LC08_L1TP_126036_20210827_20210901_02_T1/126036.tif',
    '/Users/tanzhenyu/Dataware/GeoPy/06/LC08_L1TP_127036_20210802_20210810_02_T1/127036.tif',
    '/Users/tanzhenyu/Dataware/GeoPy/06/LC08_L1TP_127037_20210802_20210810_02_T1/127037.tif'
]
boundary = '/Users/tanzhenyu/Dataware/GeoPy/06/XiAn.shp'
output = '/Users/tanzhenyu/Dataware/GeoPy/06/XiAn-202108-WGS84.tif'

# 使用Warp()函数实现多幅影像的拼接和裁减和重投影
# 第一个参数是输出结果文件，第二个参数是输入数据，其他参数都是可选参数
# cutlineDSName设置裁减范围文件，cropToCutline指定裁减到给定的范围之内
# dstSRS设置输出文件空间参考
gdal.Warp(output, inputs, cutlineDSName=boundary, cropToCutline=True,
          srcNodata=0, dstNodata=0, dstSRS='EPSG:4326')
# gdal.Warp()函数的返回结果是gdal.Dataset，指向结果数据集所在的内存对象，我们可以基于该对象做进一步操作

<osgeo.gdal.Dataset; proxy of <Swig Object of type 'GDALDatasetShadow *' at 0x15aa0e5d0> >